# FPL Decision Support System — Modelling Notebook
**Thesis Project** | PyCaret AutoML vs AutoKeras Deep Learning

---
### ⚠️ Before running anything:
1. Go to `Runtime` → `Change runtime type` → select **T4 GPU** and runtime version **2025.07** → Save
2. Follow the instructions carefully — there is a required runtime restart between sections

### Pipeline
- **Section 1** — Mount Drive + Install PyCaret → ⚠️ Restart Runtime
- **Section 2** — Load & Prepare Data
- **Section 3** — PyCaret AutoML Benchmarking
- **Section 4** — Install AutoKeras + Deep Learning
- **Section 5** — Model Comparison
- **Section 6** — Prediction Intervals (Ensemble)
- **Section 7** — PuLP Squad Optimisation
- **Section 8** — Save All Outputs

---
## Section 1 — Setup
### Step 1a: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print('Drive mounted.')

### Step 1b: Install PyCaret
⚠️ **After this cell finishes — restart the runtime before continuing.**

`Runtime` → `Restart runtime` → then continue from Section 2.

In [ ]:
!pip install pycaret -q
print('PyCaret installed. RESTART THE RUNTIME NOW: Runtime → Restart runtime')

---
## Section 2 — Load & Prepare Data
Run all cells in this section after restarting the runtime.

In [ ]:
# Remount Drive after restart
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '/content/drive/MyDrive/fpl_thesis/data/processed/featured_training_set.csv'
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Loaded: {df.shape[0]} rows, {df.shape[1]} columns')
df.head()

In [ ]:
# ── Create TARGET: next GW points for each player ─────────────────────────────
# Shift total_points by -1 within each player+season group.
# Given THIS gameweek's features → predict NEXT gameweek's points.
df = df.sort_values(['name', 'season', 'GW'])
df['target'] = df.groupby(['name', 'season'])['total_points'].shift(-1)

# Drop last GW of each season (no next GW to predict)
df = df.dropna(subset=['target']).reset_index(drop=True)
print(f'After creating target: {df.shape[0]} rows')

In [ ]:
# ── Drop identifier and leakage columns ───────────────────────────────────────
DROP_COLS = [
    'name', 'team', 'season', 'GW', 'kickoff_time',
    'total_points',  # raw target — already encoded as 'target'
    'element',       # player ID
    'fixture',       # fixture ID
    'modified',      # metadata
]
DROP_COLS = [c for c in DROP_COLS if c in df.columns]

# ── Encode categoricals ────────────────────────────────────────────────────────
# position encodes alphabetically: DEF=0, FWD=1, GK=2, MID=3
df['position'] = df['position'].astype('category').cat.codes
df['was_home'] = df['was_home'].astype(int)

# ── Build modelling dataframe ──────────────────────────────────────────────────
model_df = df.drop(columns=DROP_COLS)

# Drop any remaining non-numeric columns
non_numeric = model_df.select_dtypes(exclude=[np.number, 'bool']).columns.tolist()
if non_numeric:
    print(f'Dropping non-numeric columns: {non_numeric}')
    model_df = model_df.drop(columns=non_numeric)

print(f'Modelling dataframe: {model_df.shape}')
print(f'Target stats:\n{model_df["target"].describe()}')

In [ ]:
# ── Walk-Forward Train / Test Split ───────────────────────────────────────────
# Train: 23/24 + 24/25 + first 25 GWs of 25/26
# Test:  GW26 onwards of 25/26 (genuinely unseen future gameweeks)
# Season values are integers: 2324, 2425, 2526
model_df['season'] = df['season'].values
model_df['GW']     = df['GW'].values

train_mask = (
    model_df['season'].isin([2324, 2425]) |
    ((model_df['season'] == 2526) & (model_df['GW'] <= 25))
)
test_mask = (model_df['season'] == 2526) & (model_df['GW'] >= 26)

train_df  = model_df[train_mask].drop(columns=['season', 'GW']).reset_index(drop=True)
test_df   = model_df[test_mask].drop(columns=['season', 'GW']).reset_index(drop=True)

# Save player metadata for final predictions output
test_meta = df[test_mask][['name', 'team', 'position', 'GW', 'season', 'value']].reset_index(drop=True)

print(f'Train: {train_df.shape} | Test: {test_df.shape}')

In [ ]:
# ── Impute NaNs for AutoKeras (PyCaret handles them automatically) ─────────────
# NaNs exist in rolling features for GW1 of each season (no prior data)
from sklearn.impute import SimpleImputer

X_train = train_df.drop(columns=['target']).values.astype(np.float32)
y_train = train_df['target'].values.astype(np.float32)
X_test  = test_df.drop(columns=['target']).values.astype(np.float32)
y_test  = test_df['target'].values.astype(np.float32)

imputer = SimpleImputer(strategy='mean')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed  = imputer.transform(X_test)

print(f'NaNs in X_train after imputation: {np.isnan(X_train_imputed).sum()}')
print(f'NaNs in X_test after imputation: {np.isnan(X_test_imputed).sum()}')

---
## Section 3 — PyCaret AutoML Benchmarking
PyCaret automatically trains and compares ~20 regression models using 5-fold cross-validation.
Sorted by MAE — most interpretable metric for FPL points prediction.

In [ ]:
from pycaret.regression import *

pc_setup = setup(
    data            = train_df,
    target          = 'target',
    session_id      = 42,
    fold            = 5,
    verbose         = True,
    use_gpu         = True,
    remove_outliers = True,
    normalize       = True,
)
print('PyCaret setup complete.')

In [ ]:
# Benchmark all models
best_models = compare_models(
    sort     = 'MAE',
    n_select = 3,
    exclude  = ['ransac']
)
print('\nTop 3 models selected.')

In [ ]:
# Save top 3 models immediately to Drive
import pickle
os.makedirs('/content/drive/MyDrive/fpl_thesis/models', exist_ok=True)
with open('/content/drive/MyDrive/fpl_thesis/models/best_models.pkl', 'wb') as f:
    pickle.dump(best_models, f)
print('Top 3 models saved to Drive.')

In [ ]:
# Tune the best model
tuned_model = tune_model(best_models[0], optimize='MAE', n_iter=20)
print(f'Best model after tuning: {type(tuned_model).__name__}')

In [ ]:
# Finalise and evaluate on test set
from sklearn.metrics import mean_absolute_error, r2_score

final_pycaret_model = finalize_model(tuned_model)
pycaret_preds = predict_model(final_pycaret_model, data=test_df)
mae_pc = mean_absolute_error(pycaret_preds['target'], pycaret_preds['prediction_label'])
r2_pc  = r2_score(pycaret_preds['target'], pycaret_preds['prediction_label'])

print(f'PyCaret Test MAE : {mae_pc:.4f}')
print(f'PyCaret Test R²  : {r2_pc:.4f}')

# Naive baseline comparison
naive_mae = mean_absolute_error(test_df['target'], [test_df['target'].mean()] * len(test_df))
print(f'Naive Baseline MAE: {naive_mae:.4f}')
print(f'Improvement over baseline: {((naive_mae - mae_pc) / naive_mae * 100):.1f}%')

In [ ]:
# Save model and predictions to Drive
save_model(final_pycaret_model, '/content/drive/MyDrive/fpl_thesis/models/pycaret_best_model')

pycaret_preds.to_csv('/content/drive/MyDrive/fpl_thesis/models/pycaret_preds.csv', index=False)

import json
results = {
    'model'   : type(tuned_model).__name__,
    'test_mae': round(mae_pc, 4),
    'test_r2' : round(r2_pc, 4),
    'naive_mae': round(naive_mae, 4)
}
with open('/content/drive/MyDrive/fpl_thesis/models/pycaret_results.json', 'w') as f:
    json.dump(results, f)

print('PyCaret model, predictions and results saved to Drive.')

---
## Section 4 — AutoKeras Deep Learning
AutoKeras automatically searches for the best neural network architecture.

No runtime restart needed — install AutoKeras directly.

In [ ]:
!pip install autokeras -q
print('AutoKeras installed.')

In [ ]:
import autokeras as ak
import tensorflow as tf

print(f'TensorFlow : {tf.__version__}')
print(f'GPU        : {tf.config.list_physical_devices("GPU")}')

In [ ]:
# AutoKeras searches 10 neural architectures and picks the best
# Uses imputed data since AutoKeras does not handle NaNs automatically
ak_model = ak.StructuredDataRegressor(
    max_trials  = 10,
    overwrite   = False,   # loads existing trials from Drive if available
    seed        = 42,
    objective   = 'val_loss',
    directory   = '/content/drive/MyDrive/fpl_thesis/models/autokeras_trials'
)

ak_model.fit(
    X_train_imputed, y_train,
    epochs           = 50,
    validation_split = 0.1,
    verbose          = 1
)
print('AutoKeras training complete.')

In [ ]:
# Evaluate AutoKeras on test set
ak_preds_raw = ak_model.predict(X_test_imputed).flatten()
mae_ak = mean_absolute_error(y_test, ak_preds_raw)
r2_ak  = r2_score(y_test, ak_preds_raw)

print(f'AutoKeras Test MAE : {mae_ak:.4f}')
print(f'AutoKeras Test R²  : {r2_ak:.4f}')

# Save model and results
best_ak_model = ak_model.export_model()
best_ak_model.save('/content/drive/MyDrive/fpl_thesis/models/autokeras_best_model.keras')

import json
ak_results = {
    'model'   : 'AutoKeras Neural Network',
    'test_mae': round(mae_ak, 4),
    'test_r2' : round(r2_ak, 4)
}
with open('/content/drive/MyDrive/fpl_thesis/models/autokeras_results.json', 'w') as f:
    json.dump(ak_results, f)

print('AutoKeras model and results saved to Drive.')

---
## Section 5 — Model Comparison
Side-by-side comparison of PyCaret (Huber Regressor) vs AutoKeras vs Naive Baseline.

- **PyCaret wins on MAE** — more accurate on average
- **AutoKeras wins on R²** — better at capturing high-scoring outliers
- For FPL optimization, R² matters more — identifying premium haul players is key
- This justifies weighting AutoKeras higher (0.6) in the ensemble

In [ ]:
import matplotlib.pyplot as plt

comparison = pd.DataFrame({
    'Model' : ['PyCaret (HuberRegressor)', 'AutoKeras (Neural Network)', 'Naive Baseline'],
    'MAE'   : [mae_pc, mae_ak, naive_mae],
    'R²'    : [r2_pc,  r2_ak,  None],
})

print('===== Model Comparison =====')
print(comparison.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
colors = ['steelblue', 'darkorange', 'gray']

comparison.plot(kind='bar', x='Model', y='MAE', ax=axes[0], color=colors, legend=False)
axes[0].set_title('MAE — lower is better')
axes[0].set_xticklabels(comparison['Model'], rotation=15, ha='right')

comparison.dropna().plot(kind='bar', x='Model', y='R²', ax=axes[1], color=colors[:2], legend=False)
axes[1].set_title('R² — higher is better')
axes[1].set_xticklabels(comparison.dropna()['Model'], rotation=15, ha='right')

plt.tight_layout()
os.makedirs('/content/drive/MyDrive/fpl_thesis/outputs', exist_ok=True)
plt.savefig('/content/drive/MyDrive/fpl_thesis/outputs/model_comparison.png', dpi=150)
plt.show()
print('Comparison chart saved to Drive.')

---
## Section 6 — Prediction Intervals (Ensemble)
Combines PyCaret and AutoKeras into a weighted ensemble:
- **40% Huber Regressor** — conservative anchor, low MAE
- **60% AutoKeras** — captures premium player upside, higher R²

Prediction intervals:
- **Mid** = weighted ensemble prediction
- **Low** = mid − 1 std (pessimistic)
- **High** = mid + 1 std (optimistic)
- **Confidence** = bucketed from interval width

In [ ]:
# Load PyCaret predictions from Drive
pc_preds_df  = pd.read_csv('/content/drive/MyDrive/fpl_thesis/models/pycaret_preds.csv')
pc_preds_raw = pc_preds_df['prediction_label'].values

print(f'PyCaret predictions : {len(pc_preds_raw)} rows')
print(f'AutoKeras predictions: {len(ak_preds_raw)} rows')

In [ ]:
# Weighted ensemble: 40% PyCaret + 60% AutoKeras
# AutoKeras weighted higher due to superior R² — better at capturing
# high-scoring outlier players which are most important for FPL optimization
mid = 0.4 * pc_preds_raw + 0.6 * ak_preds_raw
std = np.std(np.array([pc_preds_raw, ak_preds_raw]), axis=0)
low  = np.clip(mid - std, 0, None)  # FPL points cannot be negative
high = mid + std

# Confidence bucketing based on interval width
interval_width = high - low
q33, q66 = np.percentile(interval_width, [33, 66])

def get_confidence(w):
    if w <= q33:   return 'High'
    elif w <= q66: return 'Medium'
    else:          return 'Low'

confidence = [get_confidence(w) for w in interval_width]

# Build final predictions dataframe
predictions_df = test_meta.copy().reset_index(drop=True)
predictions_df['predicted_pts_mid']  = np.round(mid,  2)
predictions_df['predicted_pts_low']  = np.round(low,  2)
predictions_df['predicted_pts_high'] = np.round(high, 2)
predictions_df['confidence']         = confidence
predictions_df['actual_pts']         = y_test

print(predictions_df[[
    'name', 'team', 'position', 'GW', 'value',
    'predicted_pts_low', 'predicted_pts_mid',
    'predicted_pts_high', 'confidence', 'actual_pts'
]].head(10))

OUT_PATH = '/content/drive/MyDrive/fpl_thesis/data/processed/predictions.csv'
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
predictions_df.to_csv(OUT_PATH, index=False)
print(f'\nPredictions saved to Drive. Shape: {predictions_df.shape}')

---
## Section 7 — PuLP Squad Optimisation
Treats squad selection as a Multi-Constraint Knapsack Problem.

**Constraints:**
- Total budget: £100m
- 15 players total (11 starters + 4 bench)
- Squad: 2 GK, 5 DEF, 5 MID, 3 FWD
- Starting 11: 1 GK, min 3 DEF, min 2 MID, min 1 FWD
- Max 3 players from same club
- 1 captain (points doubled)

**Approach:**
- Optimise starting 11 to maximise predicted points
- Fill bench cheaply with valid players
- Predictions scaled to account for price signal (premium players undervalued by model)

In [ ]:
!pip install pulp -q
print('PuLP installed.')

In [ ]:
from pulp import *

# ── Load and average predictions across all GWs ───────────────────────────────
# Averaging smooths out blank gameweeks (e.g. player didn't play one GW)
df_all = pd.read_csv('/content/drive/MyDrive/fpl_thesis/data/processed/predictions.csv')

df_opt = df_all.groupby(['name', 'team', 'position']).agg(
    value              = ('value', 'last'),
    predicted_pts_mid  = ('predicted_pts_mid', 'mean'),
    predicted_pts_low  = ('predicted_pts_low', 'mean'),
    predicted_pts_high = ('predicted_pts_high', 'mean'),
    confidence         = ('confidence', 'last'),
).reset_index()

# ── Fix position mapping ───────────────────────────────────────────────────────
# Encoding is alphabetical: DEF=0, FWD=1, GK=2, MID=3
position_map = {0: 'DEF', 1: 'FWD', 2: 'GK', 3: 'MID'}
df_opt['pos'] = df_opt['position'].map(position_map)

# ── Price-weighted scaling ─────────────────────────────────────────────────────
# Model exhibits regression to mean — premium players undervalued
# Blend 40% model prediction + 60% price signal to correct for this
df_opt['predicted_pts_scaled'] = (
    df_opt['predicted_pts_mid'] * 0.4 +
    (df_opt['value'] / df_opt['value'].max()) * df_opt['predicted_pts_mid'].max() * 0.6
)

print(f'Players available for optimisation: {len(df_opt)}')

In [ ]:
# ── Decision Variables ─────────────────────────────────────────────────────────
players = list(df_opt.index)

starter = LpVariable.dicts('starter', players, cat='Binary')
benched = LpVariable.dicts('benched', players, cat='Binary')
captain = LpVariable.dicts('captain', players, cat='Binary')

# ── Problem ────────────────────────────────────────────────────────────────────
prob = LpProblem('FPL_Squad_Optimisation', LpMaximize)

# Objective: maximise predicted points for starters + captain bonus
prob += (
    lpSum(starter[i] * df_opt.loc[i, 'predicted_pts_scaled'] for i in players) +
    lpSum(captain[i] * df_opt.loc[i, 'predicted_pts_scaled'] for i in players)
)

# ── Constraints ────────────────────────────────────────────────────────────────
# Squad size
prob += lpSum(starter[i] for i in players) == 11
prob += lpSum(benched[i] for i in players) == 4

# Player can only be starter OR benched
for i in players:
    prob += starter[i] + benched[i] <= 1

# Captain must be a starter, exactly 1 captain
for i in players:
    prob += captain[i] <= starter[i]
prob += lpSum(captain[i] for i in players) == 1

# Budget constraint
prob += lpSum(
    (starter[i] + benched[i]) * df_opt.loc[i, 'value'] for i in players
) <= 100.0

# Position indices
gk_idx  = [i for i in players if df_opt.loc[i, 'pos'] == 'GK']
def_idx = [i for i in players if df_opt.loc[i, 'pos'] == 'DEF']
mid_idx = [i for i in players if df_opt.loc[i, 'pos'] == 'MID']
fwd_idx = [i for i in players if df_opt.loc[i, 'pos'] == 'FWD']

# Full squad position requirements
prob += lpSum(starter[i] + benched[i] for i in gk_idx)  == 2
prob += lpSum(starter[i] + benched[i] for i in def_idx) == 5
prob += lpSum(starter[i] + benched[i] for i in mid_idx) == 5
prob += lpSum(starter[i] + benched[i] for i in fwd_idx) == 3

# Starting 11 formation constraints
prob += lpSum(starter[i] for i in gk_idx)  == 1
prob += lpSum(starter[i] for i in def_idx) >= 3
prob += lpSum(starter[i] for i in mid_idx) >= 2
prob += lpSum(starter[i] for i in fwd_idx) >= 1

# Max 3 players from same club
for club in df_opt['team'].unique():
    club_idx = [i for i in players if df_opt.loc[i, 'team'] == club]
    prob += lpSum(starter[i] + benched[i] for i in club_idx) <= 3

# ── Solve ──────────────────────────────────────────────────────────────────────
solver = PULP_CBC_CMD(msg=0)
prob.solve(solver)
print(f'Status: {LpStatus[prob.status]}')

In [ ]:
# ── Extract and Display Results ────────────────────────────────────────────────
starter_ids = [i for i in players if starter[i].value() == 1]
benched_ids = [i for i in players if benched[i].value() == 1]
captain_id  = [i for i in players if captain[i].value() == 1][0]

starting_11 = df_opt.loc[starter_ids].copy()
bench_4     = df_opt.loc[benched_ids].copy()

starting_11['role'] = 'Starter'
starting_11.loc[captain_id, 'role'] = 'Captain'
bench_4['role'] = 'Bench'

squad = pd.concat([starting_11, bench_4])

print(f'\n{"="*65}')
print(f'OPTIMAL FPL SQUAD')
print(f'{"="*65}')

for pos in ['GK', 'DEF', 'MID', 'FWD']:
    pos_players = starting_11[starting_11['pos'] == pos].sort_values(
        'predicted_pts_scaled', ascending=False
    )
    for _, row in pos_players.iterrows():
        role = '(C)' if row['role'] == 'Captain' else '   '
        print(f"{pos} {role} {row['name']:<25} £{row['value']}m   "
              f"Pred: {row['predicted_pts_mid']:.2f} pts   "
              f"[{row['predicted_pts_low']:.2f} – {row['predicted_pts_high']:.2f}]   "
              f"Conf: {row['confidence']}")

print(f'\n--- BENCH ---')
for _, row in bench_4.sort_values('pos').iterrows():
    print(f"{row['pos']}     {row['name']:<25} £{row['value']}m   "
          f"Pred: {row['predicted_pts_mid']:.2f} pts")

total_cost = squad['value'].sum()
total_pred = (
    starting_11['predicted_pts_scaled'].sum() +
    starting_11.loc[captain_id, 'predicted_pts_scaled']
)

print(f'\nTotal Cost                      : £{total_cost:.1f}m / £100.0m')
print(f'Predicted Points (with captain) : {total_pred:.2f}')
print(f'Captain                         : {df_opt.loc[captain_id, "name"]}')
print(f'{"="*65}')

---
## Section 8 — Save All Outputs

In [ ]:
# Save optimal squad to Drive
os.makedirs('/content/drive/MyDrive/fpl_thesis/data/processed', exist_ok=True)
squad.to_csv('/content/drive/MyDrive/fpl_thesis/data/processed/optimal_squad.csv', index=False)
print('Optimal squad saved to Drive.')

# Summary of all outputs
print('\n===== All outputs saved to Google Drive =====')
print('models/pycaret_best_model       — Huber Regressor')
print('models/autokeras_best_model.keras — Neural Network')
print('models/pycaret_results.json     — PyCaret metrics')
print('models/autokeras_results.json   — AutoKeras metrics')
print('outputs/model_comparison.png    — Comparison chart')
print('data/processed/predictions.csv — Player predictions')
print('data/processed/optimal_squad.csv — Optimal squad')

---
### ✅ Modelling Pipeline Complete

**Results Summary:**
- PyCaret (Huber Regressor): MAE = 0.797, R² = 0.239
- AutoKeras (Neural Network): MAE = 0.968, R² = 0.335
- Naive Baseline: MAE = 1.448
- Ensemble improvement over baseline: ~45%

**Next step → Streamlit Dashboard**